In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import sys

common_modules_path = (
    "/content/drive/MyDrive/ai-projects/2026-03-transfer-learning-resnet-50"
)
sys.path.append(common_modules_path)
print(f"Updated sys.path: {sys.path}")

Updated sys.path: ['/content', '/env/python', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/usr/local/lib/python3.12/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.12/dist-packages/IPython/extensions', '/root/.ipython', '/content/drive/MyDrive/ai-projects/2026-03-transfer-learning-resnet-50']


# PyTorch ResNet-50 Implementation

In [1]:
import json
import os
import time

import numpy as np  # ✅ ADDED
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import models, transforms

import config
from base_loader import BaseDataLoader
from torch_dataset import TorchImageDataset

# Set seed for reproducibility
torch.manual_seed(config.SEED)

# At the VERY TOP of your script, before any CUDA operations
import torch

torch.backends.cudnn.benchmark = True  # ← ADD THIS

In [ ]:
result_dir = config.PROJECT_ROOT_DIR + "/torch-results"
os.makedirs(result_dir, exist_ok=True)

## Load Dataset

In [ ]:
# ============================================
# Transforms
# ============================================
train_transform = transforms.Compose(
    [
        transforms.Resize(config.IMG_SIZE),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.Normalize(mean=config.MEAN, std=config.STD),
    ]
)

val_transform = transforms.Compose(
    [
        transforms.Resize(config.IMG_SIZE),
        transforms.Normalize(mean=config.MEAN, std=config.STD),
    ]
)

# ============================================
# DataLoaders
# ============================================
train_base = BaseDataLoader(
    config.COVIDQU_PATH, split="Train", is_train_shuffle=True, seed=config.SEED
)
val_base = BaseDataLoader(
    config.COVIDQU_PATH, split="Val", is_train_shuffle=False, seed=config.SEED
)
test_base = BaseDataLoader(
    config.COVIDQU_PATH, split="Test", is_train_shuffle=False, seed=config.SEED
)

train_ds = TorchImageDataset(train_base, transform=train_transform)
val_ds = TorchImageDataset(val_base, transform=val_transform)
test_ds = TorchImageDataset(test_base, transform=val_transform)

# Use pin_memory for faster GPU transfer
train_loader = DataLoader(
    train_ds, batch_size=config.BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True
)
val_loader = DataLoader(
    val_ds, batch_size=config.BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True
)
test_loader = DataLoader(
    test_ds, batch_size=config.BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True
)

## Build Model

In [ ]:
# ============================================
# Model
# ============================================
def build_model(num_classes):
    model = models.resnet50(weights="ResNet50_Weights.IMAGENET1K_V1")

    # Freeze all layers
    for param in model.parameters():
        param.requires_grad = False

    # Replace classifier head
    num_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Linear(num_features, 256),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(256, num_classes),
    )
    return model


num_classes = len(train_base.class_to_idx)
model = build_model(num_classes)
model = model.to(config.DEVICE)  # ✅ ADDED

# Count trainable parameters (✅ MOVED here — after model is built)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable_params:,}")

Trainable parameters: 525,315


## Layer-by-layer dump (manual comparison)

Run the **next code cell** and copy the printed table next to the same section in **`tf_model.ipynb`**. PyTorch lists **`named_modules()`** (depth-first). Compare **parameter counts** and **block names** to the Keras **nested ResNet** + **head** output—not row index.

In [ ]:
# --- PyTorch: layer-by-layer text dump (for side-by-side manual check) ---

print("=" * 110)
print("Top-level children (`model.named_children()`)")
print("=" * 110)
print(f"{'name':<22}  {'class':<36}  {'all_params_in_subtree':>20}")
print("-" * 110)
for name, mod in model.named_children():
    subtree = sum(p.numel() for p in mod.parameters(recurse=True))
    print(f"{name:<22}  {mod.__class__.__name__:<36}  {subtree:20,d}")
print("=" * 110)

print()
print("=" * 110)
print("All modules with direct parameters OR no children (leaf modules)")
print("=" * 110)
print(f"{'name':<52}  {'class':<28}  {'direct_params':>14}")
print("-" * 110)
for name, mod in model.named_modules():
    if name == "":
        continue
    direct = sum(p.numel() for p in mod.parameters(recurse=False))
    children = list(mod.children())
    if direct > 0 or len(children) == 0:
        print(f"{name:<52}  {mod.__class__.__name__:<28}  {direct:14,d}")
print("=" * 110)

print()
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total:,}  |  Trainable: {trainable:,}")

## Loss, Optimizer, Scheduler

In [ ]:
# ============================================
# Loss, Optimizer, Scheduler
# ============================================
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(
    model.parameters(), lr=config.LEARNING_RATE, momentum=0.9, weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.1, patience=3
)

## Training Functions

In [ ]:
# ============================================
# Training Functions
# ============================================
from tqdm import tqdm  # Install: pip install tqdm


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    # Add progress bar
    pbar = tqdm(loader, desc="Training")
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        # Update progress bar
        pbar.set_postfix({"loss": f"{running_loss / (total / len(loader)):.4f}"})

    epoch_loss = running_loss / len(loader)
    epoch_acc = 100.0 * correct / total
    return epoch_loss, epoch_acc


def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / len(loader)
    epoch_acc = 100.0 * correct / total
    return epoch_loss, epoch_acc

## Training Loop

In [ ]:
# Verify GPU is being used
print(f"Device: {config.DEVICE}")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    # Clear cache
    torch.cuda.empty_cache()

Device: cuda
GPU available: True
GPU name: Tesla T4


In [ ]:
# ============================================
# Training Loop (CSV log + early stopping, aligned with tf_train.EarlyStopping)
# ============================================
import csv

# Same settings as tf_train.get_callbacks() -> EarlyStopping
EARLY_STOPPING_PATIENCE = 5
EARLY_STOPPING_MONITOR = "val_loss"  # lower is better
CKPT_BEST_VAL_LOSS = result_dir + "/torch_best_val_loss.pth"
CKPT_BEST_VAL_ACC = result_dir + "/torch_best_model.pth"

best_val_acc = 0.0
best_val_loss = float("inf")
epochs_no_improve = 0

history = {
    "epoch": [],
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
    "time": [],
}

csv_path = result_dir + "/torch_training_log.csv"
start_time = time.time()

with open(csv_path, "w", newline="", encoding="utf-8") as f_csv:
    csv_writer = csv.writer(f_csv)
    csv_writer.writerow(
        [
            "epoch",
            "train_loss",
            "train_acc",
            "val_loss",
            "val_acc",
            "epoch_time_sec",
            "lr",
        ]
    )
    f_csv.flush()

    for epoch in range(config.EPOCHS):
        epoch_start = time.time()

        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, config.DEVICE
        )
        val_loss, val_acc = validate(model, val_loader, criterion, config.DEVICE)
        scheduler.step(val_loss)

        epoch_time = time.time() - epoch_start
        lr_now = optimizer.param_groups[0]["lr"]

        # Best val accuracy checkpoint (matches TF ModelCheckpoint on val_accuracy)
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), CKPT_BEST_VAL_ACC)

        # Early stopping: best val_loss (matches TF EarlyStopping monitor=val_loss)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), CKPT_BEST_VAL_LOSS)
        else:
            epochs_no_improve += 1

        history["epoch"].append(epoch + 1)
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["time"].append(epoch_time)

        csv_writer.writerow(
            [epoch + 1, train_loss, train_acc, val_loss, val_acc, epoch_time, lr_now]
        )
        f_csv.flush()

        print(
            f"Epoch {epoch + 1}/{config.EPOCHS} - {epoch_time:.1f}s - loss: {train_loss:.4f} - acc: {train_acc:.2f}% - val_loss: {val_loss:.4f} - val_acc: {val_acc:.2f}%"
        )

        if epochs_no_improve >= EARLY_STOPPING_PATIENCE:
            print(
                f"Early stopping: {EARLY_STOPPING_MONITOR} did not improve for "
                f"{EARLY_STOPPING_PATIENCE} epochs (same patience as TensorFlow)."
            )
            break

train_time = time.time() - start_time

# restore_best_weights=True equivalent: load best val_loss weights for downstream eval
if os.path.isfile(CKPT_BEST_VAL_LOSS):
    state = torch.load(CKPT_BEST_VAL_LOSS, map_location=config.DEVICE)
    model.load_state_dict(state)
    print(
        "Restored weights from best val_loss checkpoint "
        f"({CKPT_BEST_VAL_LOSS}; matches Keras EarlyStopping restore_best_weights)."
    )

# Throughput: training images processed per wall-clock second (includes validation each epoch)
_dataset_size = len(train_base)
_epochs = len(history["epoch"])
_throughput = (_dataset_size * _epochs) / train_time if train_time > 0 else 0.0
print(f"Throughput: {_throughput:.1f} images/sec")
print(f"CSV log: {csv_path}")


Training:   0%|          | 0/30 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Training: 100%|██████████| 30/30 [00:27<00:00,  1.10it/s, loss=0.1296]


Epoch 1/15 - 34.8s - loss: 0.5370 - acc: 78.94% - val_loss: 0.5147 - val_acc: 81.87%


Training: 100%|██████████| 30/30 [00:30<00:00,  1.00s/it, loss=0.1256]


Epoch 2/15 - 37.5s - loss: 0.5201 - acc: 79.72% - val_loss: 0.4928 - val_acc: 81.65%


Training: 100%|██████████| 30/30 [00:28<00:00,  1.06it/s, loss=0.1200]


...

Epoch 13/15 - 36.6s - loss: 0.3892 - acc: 83.99% - val_loss: 0.4220 - val_acc: 83.48%


Training: 100%|██████████| 30/30 [00:27<00:00,  1.07it/s, loss=0.0898]


Epoch 14/15 - 36.8s - loss: 0.3718 - acc: 85.38% - val_loss: 0.3981 - val_acc: 84.87%


Training: 100%|██████████| 30/30 [00:29<00:00,  1.03it/s, loss=0.0860]


Epoch 15/15 - 36.5s - loss: 0.3562 - acc: 85.54% - val_loss: 0.3836 - val_acc: 84.98%


## Evaluation, Collect Predictions, Save Results, Summary

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

# ============================================
# Evaluation
# ============================================
test_loss, test_acc = validate(model, test_loader, criterion, config.DEVICE)
print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Accuracy (validate, %): {test_acc:.2f}%")

# ============================================
# Collect Predictions
# ============================================
def collect_predictions(model, loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            preds = outputs.argmax(dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_probs.extend(probs.cpu().numpy())

    return np.array(all_labels), np.array(all_preds), np.array(all_probs)


y_true, y_pred, y_proba = collect_predictions(model, test_loader, config.DEVICE)
class_names = list(test_base.class_to_idx.keys())

# ============================================
# Save predictions (same layout as TensorFlow npz)
# ============================================
np.savez(
    result_dir + "/torch_predictions.npz",
    y_true=y_true,
    y_pred=y_pred,
    y_pred_proba=y_proba,
)
print(f"✅ Saved predictions to {result_dir}/torch_predictions.npz")

# ============================================
# Full test metrics JSON (mirrors tf_test_metrics.json)
# ============================================
test_metrics = {
    "framework": "PYTORCH",
    "test_accuracy": float(accuracy_score(y_true, y_pred)),
    "test_precision_macro": float(
        precision_score(y_true, y_pred, average="macro", zero_division=0)
    ),
    "test_recall_macro": float(
        recall_score(y_true, y_pred, average="macro", zero_division=0)
    ),
    "test_f1_macro": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    "test_precision_weighted": float(
        precision_score(y_true, y_pred, average="weighted", zero_division=0)
    ),
    "test_recall_weighted": float(
        recall_score(y_true, y_pred, average="weighted", zero_division=0)
    ),
    "test_f1_weighted": float(
        f1_score(y_true, y_pred, average="weighted", zero_division=0)
    ),
    "num_samples": len(y_true),
    "class_names": class_names,
}
test_metrics["per_class"] = classification_report(
    y_true, y_pred, target_names=class_names, output_dict=True, zero_division=0
)

with open(result_dir + "/torch_test_metrics.json", "w", encoding="utf-8") as f:
    json.dump(test_metrics, f, indent=4)
print(f"✅ Saved metrics to {result_dir}/torch_test_metrics.json")

cm = confusion_matrix(y_true, y_pred)
np.save(result_dir + "/torch_confusion_matrix.npy", cm)
print(f"✅ Saved confusion matrix to {result_dir}/torch_confusion_matrix.npy")

with open(result_dir + "/torch_classification_report.txt", "w", encoding="utf-8") as f:
    f.write("Classification Report - PYTORCH\n")
    f.write("=" * 50 + "\n")
    f.write(
        classification_report(
            y_true, y_pred, target_names=class_names, zero_division=0
        )
    )
print(f"✅ Saved classification report to {result_dir}/torch_classification_report.txt")

# ============================================
# Run summary JSON (aligned with tf_results.json semantics)
# ============================================
results = {
    "framework": "PyTorch",
    "test_accuracy": float(accuracy_score(y_true, y_pred)),
    "test_loss": float(test_loss),
    "training_time_seconds": train_time,
    "best_val_acc": float(best_val_acc) / 100.0,
    "epochs_completed": len(history["epoch"]),
    "throughput_images_per_sec": float(
        (len(train_base) * len(history["epoch"])) / train_time
        if train_time > 0
        else 0.0
    ),
    "final_train_loss": float(history["train_loss"][-1]),
    "final_train_acc": float(history["train_acc"][-1]) / 100.0,
    "final_val_loss": float(history["val_loss"][-1]),
    "final_val_acc": float(history["val_acc"][-1]) / 100.0,
    "test_precision_macro": test_metrics["test_precision_macro"],
    "test_recall_macro": test_metrics["test_recall_macro"],
    "test_f1_macro": test_metrics["test_f1_macro"],
    "test_precision_weighted": test_metrics["test_precision_weighted"],
    "test_recall_weighted": test_metrics["test_recall_weighted"],
    "test_f1_weighted": test_metrics["test_f1_weighted"],
    "num_samples": len(y_true),
    "class_names": class_names,
}

with open(result_dir + "/torch_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4)

with open(result_dir + "/torch_history.json", "w", encoding="utf-8") as f:
    json.dump(history, f, indent=4)

print(f"\n✅ Results saved to {result_dir}/torch_results.json")

# ============================================
# Summary (console)
# ============================================
print("\n" + "=" * 50)
print("FINAL TEST METRICS - PYTORCH")
print("=" * 50)
print(f"Accuracy:  {test_metrics['test_accuracy']:.4f}")
print(f"Precision: {test_metrics['test_precision_macro']:.4f} (macro)")
print(f"Recall:    {test_metrics['test_recall_macro']:.4f} (macro)")
print(f"F1-Score:  {test_metrics['test_f1_macro']:.4f} (macro)")
print("=" * 50)

print("\n=== PyTorch Training Complete ===")
print(f"Best Val Accuracy: {best_val_acc:.2f}%")
print(f"Test Accuracy (sklearn): {test_metrics['test_accuracy'] * 100:.2f}%")
print(f"Total Time: {train_time:.1f} seconds")
print(f"Trainable Params: {trainable_params:,}")



Test Loss: 0.3205
Test Accuracy: 88.08%

✅ Results saved to /content/drive/MyDrive/ai-projects/2026-03-transfer-learning-resnet-50/torch-results/torch_results.json

=== PyTorch Training Complete ===
Best Val Accuracy: 85.30%
Test Accuracy: 88.08%
Total Time: 566.2 seconds
Trainable Params: 525,315


In [ ]:
# ============================================
# QUICK ANALYSIS - Run immediately after evaluation
# Same structure as tf_model.ipynb: convergence, overfitting, efficiency, verdict
# ============================================

print("\n" + "=" * 60)
print("QUICK ANALYSIS - PyTorch Run")
print("=" * 60)

# 0-1 accuracies to match Keras history semantics in the TF notebook
qa_test_acc = float(test_metrics["test_accuracy"])
qa_test_loss = float(test_loss)
val_acc = [float(x) / 100.0 for x in history["val_acc"]]
train_acc_hist = [float(x) / 100.0 for x in history["train_acc"]]
n_epochs = len(history["epoch"])

# 1. Final Performance
print("\n📊 FINAL PERFORMANCE:")
print(f"   Test Accuracy:  {qa_test_acc:.4f} ({qa_test_acc * 100:.2f}%)")
print(f"   Test Loss:      {qa_test_loss:.4f}")
print(f"   Best Val Acc:   {max(val_acc):.4f}")
print(f"   Final Val Acc:  {val_acc[-1]:.4f}")

# 2. Improvement Over Random Baseline
num_classes = len(test_base.class_to_idx)
random_baseline = 1 / num_classes
improvement = (qa_test_acc - random_baseline) * 100
print("\n📈 IMPROVEMENT:")
print(f"   Random baseline: {random_baseline:.4f} ({random_baseline * 100:.1f}%)")
print(f"   Improvement:     +{improvement:.1f} percentage points")

# 3. Convergence Check
best_val = max(val_acc)
print("\n🎯 CONVERGENCE STATUS:")
if val_acc[-1] == best_val:
    print("   ✅ Still improving (best at last epoch)")
elif val_acc[-1] >= 0.98 * best_val:
    print("   ✅ Plateaued near peak (last epoch within 2% of best)")
else:
    print(
        f"   ⚠️ May need more epochs (last epoch {val_acc[-1]:.4f} vs best {best_val:.4f})"
    )

# 4. Overfitting Check
train_acc_final = train_acc_hist[-1]
val_acc_final = val_acc[-1]
gap = train_acc_final - val_acc_final

print("\n🔍 OVERFITTING CHECK:")
print(f"   Train accuracy:  {train_acc_final:.4f}")
print(f"   Val accuracy:    {val_acc_final:.4f}")
print(f"   Gap:             {gap:+.4f}")
if gap < 0.05:
    print("   ✅ No significant overfitting (gap < 5%)")
elif gap < 0.1:
    print("   ⚠️ Moderate overfitting (gap 5-10%)")
else:
    print("   ❌ Significant overfitting (gap > 10%)")

# 5. Training Efficiency
print("\n⚡ EFFICIENCY:")
print(f"   Total training:  {train_time:.1f} seconds ({train_time / 60:.2f} minutes)")
print(f"   Epochs:          {n_epochs}")
if n_epochs > 0:
    print(f"   Avg per epoch:   {train_time / n_epochs:.1f} seconds")

# 6. Quick Verdict
print("\n✅ QUICK VERDICT:")
if qa_test_acc >= 0.70:
    print(
        f"   Model achieved {qa_test_acc * 100:.1f}% — Good for frozen transfer learning"
    )
elif qa_test_acc >= 0.60:
    print(f"   Model achieved {qa_test_acc * 100:.1f}% — Acceptable, could improve")
else:
    print(
        f"   Model achieved {qa_test_acc * 100:.1f}% — Below expectations, check pipeline"
    )

# 7. Save quick summary to text file
quick_summary = f"""
========================================
QUICK SUMMARY - PyTorch
========================================
Test Accuracy:     {qa_test_acc:.4f} ({qa_test_acc * 100:.2f}%)
Test Loss:         {qa_test_loss:.4f}
Best Val Acc:      {max(val_acc):.4f}
Final Val Acc:     {val_acc[-1]:.4f}
Random Baseline:   {random_baseline:.4f}
Improvement:       +{improvement:.1f}%

Training Time:     {train_time:.1f} seconds
Epochs:            {n_epochs}
Gap (Train-Val):   {gap:+.4f}

Status:            {"✅ Good" if qa_test_acc >= 0.70 else "⚠️ Acceptable" if qa_test_acc >= 0.60 else "❌ Needs work"}
========================================
"""

with open(result_dir + "/torch_quick_summary.txt", "w", encoding="utf-8") as f:
    f.write(quick_summary)

print("\n📁 Quick summary saved to:", result_dir + "/torch_quick_summary.txt")
print("=" * 60)


## Goal

Save model weights at best validation accuracy, log per-epoch metrics to JSON, compute full test metrics (accuracy, F1, confusion matrix), and save all predictions — then create comparison plots and tables to evaluate both frameworks side by side.